In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Load Results

In [ ]:
seeds = [42, 67, 99, 70, 73]

datasets = {
    "EchoNext": {
        72475: "runs-echonext",
        32768: "runs-echonext-32k",
        16384: "runs-echonext-16k",
        8192: "runs-echonext-8k",
        4096: "runs-echonext-4k",
        2048: "runs-echonext-2k",
        1024: "runs-echonext-1k",
        512: "runs-echonext-512",
        256: "runs-echonext-256",
    },
    "MIMIC-IV-ECG": {
        78470: "runs-mimic",
        32768: "runs-mimic-32k",
        16384: "runs-mimic-16k",
        8192: "runs-mimic-8k",
        4096: "runs-mimic-4k",
        2048: "runs-mimic-2k",
        1024: "runs-mimic-1k",
        512: "runs-mimic-512",
        256: "runs-mimic-256",
    },
    "CODE-15%": {
        74112: "runs-code15",
        32768: "runs-code15-32k",
        16384: "runs-code15-16k",
        8192: "runs-code15-8k",
        4096: "runs-code15-4k",
        2048: "runs-code15-2k",
        1024: "runs-code15-1k",
        512: "runs-code15-512",
        256: "runs-code15-256",
    },
    "PTB-XL": {
        17418: "runs-ptbxl",
        8722: "runs-ptbxl-8k",
        4356: "runs-ptbxl-4k",
        2175: "runs-ptbxl-2k",
        1091: "runs-ptbxl-1k",
        547: "runs-ptbxl-512",
        273: "runs-ptbxl-256",
    },
    "CinC Georgia": {
        8192: "runs-cinc",
        4096: "runs-cinc-4k",
        2048: "runs-cinc-2k",
        1024: "runs-cinc-1k",
        512: "runs-cinc-512",
        256: "runs-cinc-256",
    },
    "ZZU pECG": {
        8658: "runs-zzu",
        4096: "runs-zzu-4k",
        2048: "runs-zzu-2k",
        1024: "runs-zzu-1k",
        512: "runs-zzu-512",
        256: "runs-zzu-256",
    },
}

# mapping of experiment name to tuple of:
# - plotting color
# - folder name
experiments = {
    "SupProto Direct":     ("tab:green",  "supproto-direct"),
    "SupProto HEEDB":      ("tab:orange", "supproto-heedb-rila"),
    "SupProto HEEDB (FT)": ("tab:pink",   "supproto-heedb-rila-ft"),
    "ProtoSSL HEEDB":      ("tab:blue",   "protossl-heedb-pila"),
    "ProtoSSL HEEDB (FT)": ("tab:cyan",   "protossl-heedb-pila-ft"),
    ###
    "Blackbox Direct": ("tab:grey",   "blackbox-direct"),
    "ECGFounder":      ("tab:red",    "ecgfounder-logreg"),
    "ST-MEM":          ("tab:purple", "stmem-logreg"),
    "PCLR-HEEDB":      ("tab:olive", "pclr-logreg"), # new
    ###
    "SupProto HEEDB-150 (7500-proto-bank)":      ("tab:orange", "supproto-heedb-150-rila"), # new
    "SupProto HEEDB-150 (7500-proto-bank) (FT)": ("tab:pink",   "supproto-heedb-150-rila-ft"), # new
    "ProtoSSL HEEDB-150 (7500-proto-bank)":      ("tab:blue",   "protossl-heedb-150-pila"), # new
    "ProtoSSL HEEDB-150 (7500-proto-bank) (FT)": ("tab:cyan",   "protossl-heedb-150-pila-ft"), # new
    ###
    "SupProto HEEDB-150 (1050-proto-bank)":      ("tab:orange", "supproto-heedb-150-1050-rila"), # new
    "SupProto HEEDB-150 (1050-proto-bank) (FT)": ("tab:pink",   "supproto-heedb-150-1050-rila-ft"), # new
    "ProtoSSL HEEDB-150 (1050-proto-bank)":      ("tab:blue",   "protossl-heedb-150-1050-pila"), # new
    "ProtoSSL HEEDB-150 (1050-proto-bank) (FT)": ("tab:cyan",   "protossl-heedb-150-1050-pila-ft"), # new
    ###
    "ProtoSSL HEEDB (83ppl)":  ("yellow", "protossl-heedb-pila-83ppl"),
    "ProtoSSL HEEDB (PIT)":    ("blue",   "protossl-heedb-pit"),
    "ProtoSSL HEEDB (PIP)":    ("red",    "protossl-heedb-pip"),
    "ProtoSSL HEEDB (Latent)": ("orange", "protossl-heedb-latent"), # new
}

def get_palette(exp_names):
    palette = dict()
    exps = experiments
    for exp_name in exp_names:
        palette[exp_name] = exps[exp_name][0]
    return palette

In [ ]:
data = []
for seed in seeds:
    output_dir = Path(f"/opt/gpu_working/steven/protossl-ecg-outputs-rebuttal/protossl-outputs-seed{seed}")
    for ds, sizes in datasets.items():
        for size, run_dir in sizes.items():
            exps = experiments.copy()
            for exp_name, (exp_color, exp_dir) in exps.items():
                _output_dir = output_dir
                metrics_csv = _output_dir / run_dir / exp_dir / "metrics-bootstrapped-v2.csv"
                if not os.path.exists(metrics_csv):
                    continue
                metrics = pd.read_csv(metrics_csv, index_col="Label")
                multilabel = metrics.loc["Multilabel Averaged"]
                datum = {
                    "Seed": seed,
                    "Dataset": ds,
                    "Model": exp_name,
                    "Train Size": size,
                    "Multilabel (AUROC)": multilabel["AUROC"],
                    "Multilabel (AUPRC)": multilabel["AUPRC"],
                }
                if "AUROC 95% CI (lo)" in metrics.columns:
                    datum["AUROC 95% CI (lo)"] = metrics.loc["Multilabel Averaged", "AUROC 95% CI (lo)"]
                    datum["AUROC 95% CI (hi)"] = metrics.loc["Multilabel Averaged", "AUROC 95% CI (hi)"]
                data.append(datum)
results = pd.DataFrame.from_records(data)

In [ ]:
len(results)

## Plot Label Efficiency Curve

In [ ]:
def plot_lift(
    *,  # enforce kwargs
    df: pd.DataFrame, # long format df (each point to plot is a row)
    dataset: str,
    seed: int | list[int] = 42, # if list of int, average across seeds
    metric: str,
    models: list[str], # must be intentional about which models to plot
    rename: list[str] | None = None,
    baseline_model: str | None = None, # singular result to optionally plot as dashed line
    ylim: tuple[float, float] | None = None,
    xlim: tuple[float, float] | None = None,
    save_path: str | None = None,
    smooth: bool = True,
    ax: plt.Axes | None = None,
    do_label: bool = True,
) -> tuple[plt.Figure, plt.Axes]:
    if rename is not None:
        assert len(models) == len(rename)
    if baseline_model is not None and baseline_model not in models:
        models = [baseline_model] + models
        if rename is not None:
            rename = [baseline_model] + rename
    palette = get_palette(models)
    if rename is not None:
        palette = {new_name: palette[m] for new_name, m in zip(rename, models)}
        df = df.copy()
        df["Model"] = df["Model"].replace({v: k for k, v in zip(rename, models)})

    df = df[df["Dataset"] == dataset]
    if isinstance(seed, int):
        df = df[df["Seed"] == seed]
    else: # list of seeds
        df = df[df["Seed"].isin(seed)]
        df = df.groupby(["Dataset", "Model", "Train Size"]).mean().reset_index()
    # print(dataset, df[metric].min(), df[metric].max())
    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    min_size = df["Train Size"].min()
    max_size = df["Train Size"].max()
    if xlim is not None:
        min_size = min(min_size, xlim[0])
        max_size = max(max_size, xlim[1])

    if baseline_model is not None:
        mask = df["Model"] == baseline_model
        assert (
            mask.sum() == 1
        ), f"Should only have 1 entry for baseline model: {baseline_model}"
        baseline_row = df[mask].iloc[0]
        ax.hlines(
            baseline_row[metric],
            min_size,
            max_size,
            colors=palette.pop(baseline_model),
            linestyles=":",
            label=baseline_model,
        )
        df = df[~mask] # subsequent line plots should exclude baseline model

    if not smooth:
        sns.lineplot(
            df,
            x="Train Size",
            y=metric,
            hue="Model",
            palette=palette,
            hue_order=list(palette.keys()),
            marker="o",
            ax=ax,
        )
    else:
        for model, color in palette.items():
            model_data = df[df["Model"] == model]
            ax.set_xlim([2**7, 2**17])
            sns.regplot(
                model_data,
                x="Train Size",
                y=metric,
                color=color,
                marker="o",
                ax=ax,
                logx=True,
                label=model if do_label else None,
                line_kws={"zorder": 2},
                scatter_kws={"zorder": 3},
                truncate=False,
            )
    ax.set_xscale("log", base=2)
    if xlim is not None:
        ax.set_xlim(xlim)
    else:
        # tighter boundaries than default lims
        ax.set_xlim((min_size, max_size))
    ax.set_title(dataset)
    # ax.set_title(f"{dataset} {metric}")
    if do_label:
        ax.legend(loc="lower right")
    if ylim is not None:
        ax.set_ylim(ylim)
    else:
        ymin, ymax = ax.get_ylim()
        ax.set_ylim((ymin, min(ymax, 1)))
    if save_path is not None:
        assert fig is not None
        fig.tight_layout()
        fig.savefig(save_path)
    return fig, ax

### Figs to help visualize tables

In [ ]:
fig, ax = plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "SupProto Direct",
        "SupProto HEEDB-150 (7500-proto-bank)",
        "SupProto HEEDB-150 (7500-proto-bank) (FT)",
        "ProtoSSL HEEDB-150 (7500-proto-bank)",
        "ProtoSSL HEEDB-150 (7500-proto-bank) (FT)",
    ],
    seed=seeds,
    do_label=True,
)
ax.set_title("EchoNext (7500 Pretrained Prototypes, 14 PPL Downstream)")
ax.set_ylim([0.5, 0.9])

In [ ]:
fig, ax = plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "SupProto Direct",
        "SupProto HEEDB-150 (1050-proto-bank)",
        "SupProto HEEDB-150 (1050-proto-bank) (FT)",
        "ProtoSSL HEEDB-150 (1050-proto-bank)",
        "ProtoSSL HEEDB-150 (1050-proto-bank) (FT)",
    ],
    seed=seeds,
    do_label=True,
)
ax.set_title("EchoNext (1050 Pretrained Prototypes, 14 PPL Downstream)")
ax.set_ylim([0.5, 0.9])

In [ ]:
fig, ax = plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "Blackbox Direct",
        "ProtoSSL HEEDB",
        "ECGFounder",
        "ST-MEM",
        "PCLR-HEEDB",
    ],
    seed=seeds,
    do_label=True,
)
ax.set_title("EchoNext")
ax.set_ylim([0.5, 0.9])

In [ ]:
fig, ax = plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "ProtoSSL HEEDB (83ppl)",
        "ProtoSSL HEEDB (PIT)",
        "ProtoSSL HEEDB (PIP)",
        "ProtoSSL HEEDB (Latent)",
    ],
    seed=seeds,
    do_label=True,
)
ax.set_title("EchoNext")
ax.set_ylim([0.5, 0.9])

In [ ]:
print(results[(results["Model"] == "PCLR-HEEDB") & (results["Train Size"] == 72475)].to_markdown())

In [ ]:
print(results[(results["Model"] == "ProtoSSL HEEDB (83ppl)") & (results["Train Size"] == 72475)].to_markdown())

In [ ]:
fig, ax = plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "ProtoSSL HEEDB (83ppl)",
        "PCLR-HEEDB",
    ],
    seed=seeds,
    do_label=True,
)
ax.set_title("EchoNext")
ax.set_ylim([0.5, 0.9])

## Make Tables

In [ ]:
all_agg = (
    results
    .sort_values(["Model", "Dataset", "Train Size"])
    .groupby(["Dataset", "Model", "Train Size"])
    # ["Multilabel (AUROC)"].describe() # reporting std dev
    [["Multilabel (AUROC)", "AUROC 95% CI (lo)", "AUROC 95% CI (hi)"]].mean() # reporting bootstrapped CI
)
# all_agg = all_agg.reset_index()[["Dataset", "Model", "Train Size", "mean", "std"]] # reporting std dev
all_agg = all_agg.reset_index()[["Dataset", "Model", "Train Size", "Multilabel (AUROC)", "AUROC 95% CI (lo)", "AUROC 95% CI (hi)"]] # reporting bootstrapped CI

# filt_agg = all_agg[
#     all_agg["Model"].isin([
#         "SupProto Direct",
#         "SupProto HEEDB",
#         "SupProto HEEDB (FT)",
#         "ProtoSSL HEEDB",
#         "ProtoSSL HEEDB (FT)",
#     ])
# ].reset_index(drop=True)
filt_agg = all_agg

# reporting std dev
# filt_agg["val_mean"] = filt_agg["mean"].apply(lambda x: f"{x:0.3f}")
# filt_agg["val_std"] = filt_agg["std"].apply(lambda x: f"{x:.2e}")
# filt_agg["val"] = filt_agg["val_mean"] + " ± " + filt_agg["val_std"]

# report bootstrapped CI
filt_agg["val"] = (
    filt_agg["Multilabel (AUROC)"].apply(lambda x: f"{x:0.3f}")
    + " ["
    + filt_agg["AUROC 95% CI (lo)"].apply(lambda x: f"{x:0.3f}")
    + "-"
    + filt_agg["AUROC 95% CI (hi)"].apply(lambda x: f"{x:0.3f}")
    + "]"
)

pivoted = filt_agg[["Dataset", "Model", "Train Size", "val"]].pivot(columns=["Dataset", "Train Size"], index=["Model"], values="val")
pivoted = pivoted.sort_index(axis=1, level=[0, 1], ascending=[True, False])
pivoted.index.name = None

### HEEDB 150 Labels (new)

#### 1050 Pretrained Prototypes (7 PPL during pretraining for SupProto HEEDB)

In [ ]:
n1050 = pivoted.loc[
        ["ProtoSSL HEEDB-150 (1050-proto-bank) (FT)", "SupProto HEEDB-150 (1050-proto-bank) (FT)", "ProtoSSL HEEDB-150 (1050-proto-bank)", "SupProto HEEDB-150 (1050-proto-bank)", "SupProto Direct"],
        ["EchoNext"]
    ].T.copy().rename(columns=dict(zip(
        ["ProtoSSL HEEDB-150 (1050-proto-bank) (FT)", "SupProto HEEDB-150 (1050-proto-bank) (FT)", "ProtoSSL HEEDB-150 (1050-proto-bank)", "SupProto HEEDB-150 (1050-proto-bank)", "SupProto Direct"],
        ["ProtoSSL HEEDB (FT)", "SupProto HEEDB (FT)", "ProtoSSL HEEDB", "SupProto HEEDB", "SupProto Direct"]
    )))

print(n1050.to_markdown())

#### 7500 Pretrained Prototypes (50 PPL during pretraining for SupProto HEEDB)

In [ ]:
n7500 = pivoted.loc[
        ["ProtoSSL HEEDB-150 (7500-proto-bank) (FT)", "SupProto HEEDB-150 (7500-proto-bank) (FT)", "ProtoSSL HEEDB-150 (7500-proto-bank)", "SupProto HEEDB-150 (7500-proto-bank)", "SupProto Direct"],
        ["EchoNext"]
    ].T.copy().rename(columns=dict(zip(
        ["ProtoSSL HEEDB-150 (7500-proto-bank) (FT)", "SupProto HEEDB-150 (7500-proto-bank) (FT)", "ProtoSSL HEEDB-150 (7500-proto-bank)", "SupProto HEEDB-150 (7500-proto-bank)", "SupProto Direct"],
        ["ProtoSSL HEEDB (FT)", "SupProto HEEDB (FT)", "ProtoSSL HEEDB", "SupProto HEEDB", "SupProto Direct"]
    )))

print(n7500.to_markdown())

### Blackbox Baselines - including PCLR (new)

In [ ]:
print(
    pivoted.loc[
        ["PCLR-HEEDB", "ProtoSSL HEEDB", "ECGFounder", "ST-MEM", "Blackbox Direct"],
        ["EchoNext"]
    ].T.to_markdown()
)

### No Assignment (PIT & PIP) - and No Projection (Latent) (new)

In [ ]:
print(
    pivoted.loc[
        ["ProtoSSL HEEDB (83ppl)", "ProtoSSL HEEDB (PIT)", "ProtoSSL HEEDB (PIP)", "ProtoSSL HEEDB (Latent)"],
        ["EchoNext"]
    ].T.rename(columns={
        "ProtoSSL HEEDB (83ppl)": "ProtoSSL HEEDB (LAP) (83PPL)",
    }).to_markdown()
)

### ZZU full scale more prototypes in ProtoSSL pretraining

In [ ]:
print(
    pivoted.loc[
        ["ProtoSSL HEEDB", "ProtoSSL HEEDB-150 (7500-proto-bank)", "SupProto Direct"],
        ["ZZU pECG"]
    ].T.to_markdown()
)

In [ ]:
print(
    pivoted.loc[
        ["ProtoSSL HEEDB", "ProtoSSL HEEDB-150 (7500-proto-bank)", "SupProto Direct"],
        ["ZZU pECG"]
    ].T.rename(columns={
        "ProtoSSL HEEDB": "ProtoSSL HEEDB (1000-proto-bank)",
        "ProtoSSL HEEDB-150 (7500-proto-bank)": "ProtoSSL HEEDB (7500 proto-bank)"
    }).to_markdown()
)

In [ ]:
temp = filt_agg[["Dataset", "Model", "Train Size", "Multilabel (AUROC)"]].pivot(columns=["Dataset", "Train Size"], index=["Model"], values="Multilabel (AUROC)")
temp = temp.sort_index(axis=1, level=[0, 1], ascending=[True, False])
temp.index.name = None
print(
    temp.loc[
        ["ProtoSSL HEEDB", "ProtoSSL HEEDB-150 (7500-proto-bank)", "SupProto Direct"],
        ["ZZU pECG"]
    ].T.rename(columns={
        "ProtoSSL HEEDB": "ProtoSSL HEEDB (1000-proto-bank)",
        "ProtoSSL HEEDB-150 (7500-proto-bank)": "ProtoSSL HEEDB (7500 proto-bank)"
    }).to_markdown()
)